# Top K Elements (Heap)

## Pattern Overview

The **Top K Elements** pattern uses a **heap (priority queue)** to efficiently find the k largest/smallest elements, or elements ranked by a custom key.

### Python Heap Facts
- `heapq` is a **min-heap** by default
- For max-heap behavior: negate values (`heapq.heappush(h, -val)`)
- `heapq.heappush(h, x)` — O(log n)
- `heapq.heappop(h)` — O(log n), returns smallest
- `heapq.nlargest(k, iterable)` / `heapq.nsmallest(k, iterable)` — O(n log k)

---

## Three Core Templates

### Pattern 1: Top K Largest using min-heap of size k
```python
import heapq

def top_k_largest(nums, k):
    heap = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k:
            heapq.heappop(heap)  # remove smallest
    return list(heap)  # smallest of the k largest is heap[0]
```

### Pattern 2: Kth Largest
```python
def kth_largest(nums, k):
    return heapq.nlargest(k, nums)[-1]
```

### Pattern 3: Custom key (e.g., by frequency)
```python
import collections
def top_k_frequent(nums, k):
    freq = collections.Counter(nums)
    return heapq.nlargest(k, freq.keys(), key=freq.get)
```

---

## Complexity Comparison

| Method | Time | Space | Best for |
|--------|------|-------|----------|
| Sort all | O(n log n) | O(n) | Small n or need full sort |
| Heap of size k | O(n log k) | O(k) | Large n, streaming |
| heapify + k pops | O(n + k log n) | O(n) | Already have all data |
| QuickSelect | O(n) avg | O(1) | Single kth element |

---

## When to Use

| Signal | Pattern |
|--------|---------|
| "k largest/smallest" | Min-heap of size k |
| "k most/least frequent" | Custom key heap |
| "median of stream" | Two-heap (max + min) |
| "merge k sorted lists" | Min-heap with (val, list_idx) |


In [ ]:
import heapq
from collections import Counter, defaultdict
from typing import List, Optional

print("Helpers loaded.")

---
## Easy Problems (20)

### E1. Kth Largest Element in a Stream (LC 703)

> 🏢 **Asked by:** Amazon, Google, LinkedIn
Design a class that finds the kth largest element in a stream.

**Approach:** Maintain a min-heap of size k. The root is always the kth largest.
**Time:** O(log k) per add | **Space:** O(k)

In [ ]:
class KthLargest:
    def __init__(self, k, nums):
        self.k = k
        self.heap = []
        for n in nums:
            self.add(n)
    def add(self, val):
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]

kl = KthLargest(3, [4,5,8,2])
assert kl.add(3) == 4
assert kl.add(5) == 5
assert kl.add(10) == 5
assert kl.add(9) == 8
assert kl.add(4) == 8
print("All tests passed!")

### E2. Last Stone Weight (LC 1046)

> 🏢 **Asked by:** Amazon, Google
Smash the two heaviest stones; return the remaining weight.

**Approach:** Max-heap (negate values). Repeatedly pop two heaviest, push difference if nonzero.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def lastStoneWeight(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        a = -heapq.heappop(heap)
        b = -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0

assert lastStoneWeight([2,7,4,1,8,1]) == 1
assert lastStoneWeight([1]) == 1
assert lastStoneWeight([2,2]) == 0
print("All tests passed!")

### E3. K Closest Points to Origin (LC 973)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Uber
Return the k closest points to the origin (0,0).

**Approach:** Min-heap by squared distance (avoid sqrt). heapq.nsmallest or maintain heap of size k.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def kClosest(points, k):
    return heapq.nsmallest(k, points, key=lambda p: p[0]**2 + p[1]**2)

assert sorted(kClosest([[1,3],[-2,2]], 1)) == [[-2,2]]
assert len(kClosest([[3,3],[5,-1],[-2,4]], 2)) == 2
print("All tests passed!")

### E4. Take Gifts From the Richest Pile (LC 2558)

> 🏢 **Asked by:** Amazon, Google
Each second, take gifts from the richest pile equal to floor(sqrt(richest)). After k seconds, return total.

**Approach:** Max-heap. Each step: pop max, push floor(sqrt(max)), repeat k times.
**Time:** O(k log n) | **Space:** O(n)

In [ ]:
import math

def pickGifts(gifts, k):
    heap = [-g for g in gifts]
    heapq.heapify(heap)
    for _ in range(k):
        top = -heapq.heappop(heap)
        heapq.heappush(heap, -int(math.sqrt(top)))
    return -sum(heap)

assert pickGifts([25,64,9,4,100], 4) == 29
assert pickGifts([1,1,1,1], 4) == 4
print("All tests passed!")

### E5. Minimum Cost of Buying Candies With Discount (LC 2144)

> 🏢 **Asked by:** Amazon, Google
For every 3 candies (sorted desc), the cheapest is free.

**Approach:** Sort descending; every 3rd item (index 2,5,8,...) is free. Sum the rest.
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def minimumCost(cost):
    cost.sort(reverse=True)
    return sum(cost[i] for i in range(len(cost)) if (i+1) % 3 != 0)

assert minimumCost([1,2,3]) == 5
assert minimumCost([6,5,7,9,2,2]) == 23
assert minimumCost([5,5]) == 10
print("All tests passed!")

### E6. Maximum Product of Two Elements in an Array (LC 1464)

> 🏢 **Asked by:** Amazon, Google
Return (max1-1) * (max2-1) where max1 and max2 are the two largest elements.

**Approach:** Find the two largest elements with a single pass or nlargest(2).
**Time:** O(n) | **Space:** O(1)

In [ ]:
def maxProduct(nums):
    a, b = heapq.nlargest(2, nums)
    return (a-1) * (b-1)

assert maxProduct([3,4,5,2]) == 12
assert maxProduct([1,5,4,5]) == 16
assert maxProduct([3,7]) == 12
print("All tests passed!")

### E7. Find Subsequence of Length K With the Largest Sum (LC 2099)

> 🏢 **Asked by:** Amazon, Google
Return a subsequence of length k with the largest sum (preserving original order).

**Approach:** Find k largest by value keeping original indices; sort result by index to preserve order.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def maxSubsequence(nums, k):
    indexed = heapq.nlargest(k, enumerate(nums), key=lambda x: x[1])
    indexed.sort()
    return [v for _, v in indexed]

assert maxSubsequence([2,1,3,3], 2) == [3,3]
assert maxSubsequence([-1,-2,3,4], 3) == [-1,3,4]
print("All tests passed!")

### E8. Find the Kth Largest Integer in the Array (LC 1985)

> 🏢 **Asked by:** Amazon, Google
Given array of numeric strings, find kth largest as integer (no leading zeros, can be large).

**Approach:** Custom comparator: longer string = larger; same length: lexicographic compare. Use heapq with custom key.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def kthLargestNumber(nums, k):
    # Use (length, value) as sort key for numeric string comparison
    return heapq.nlargest(k, nums, key=lambda x: (len(x), x))[-1]

assert kthLargestNumber(["3","6","7","10"], 4) == "3"
assert kthLargestNumber(["2","21","12","1"], 3) == "2"
assert kthLargestNumber(["0","0"], 2) == "0"
print("All tests passed!")

### E9. Minimum Operations to Halve Array Sum (LC 2208)

> 🏢 **Asked by:** Amazon, Google
Min operations to reduce array sum by at least half; each op halves the max element.

**Approach:** Max-heap. Greedily halve the largest element until total reduction >= initial_sum/2.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def halveArray(nums):
    total = sum(nums)
    target = total / 2
    heap = [-n for n in nums]
    heapq.heapify(heap)
    ops = 0
    reduced = 0.0
    while reduced < target:
        top = -heapq.heappop(heap)
        half = top / 2
        reduced += half
        heapq.heappush(heap, -half)
        ops += 1
    return ops

assert halveArray([5,19,8,1]) == 3
assert halveArray([3,8,20]) == 3
print("All tests passed!")

### E10. Remove Stones to Minimize the Total (LC 1962)

> 🏢 **Asked by:** Amazon, Google
Perform k operations; each op: remove floor(piles[i]/2) from the largest pile.

**Approach:** Max-heap. Each operation: pop max, subtract floor(max/2), push back.
**Time:** O(k log n) | **Space:** O(n)

In [ ]:
def minStoneSum(piles, k):
    heap = [-p for p in piles]
    heapq.heapify(heap)
    for _ in range(k):
        top = -heapq.heappop(heap)
        top -= top // 2
        heapq.heappush(heap, -top)
    return -sum(heap)

assert minStoneSum([5,4,9], 2) == 12
assert minStoneSum([4,3,6,7], 3) == 12
print("All tests passed!")

### E11. Reducing Dishes (LC 1402)

> 🏢 **Asked by:** Amazon, Google
Maximize sum of satisfaction[i] * time (1-indexed) by choosing a subset of dishes.

**Approach:** Sort dishes. Include dish greedily if prefix sum (adding it) improves total. Add from largest downward.
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def maxSatisfaction(satisfaction):
    satisfaction.sort()
    total = 0
    running = 0
    for s in reversed(satisfaction):
        running += s
        if running <= 0: break
        total += running
    return total

assert maxSatisfaction([-1,-8,0,5,-9]) == 14
assert maxSatisfaction([4,3,2]) == 20
assert maxSatisfaction([-1,-4,-5]) == 0
print("All tests passed!")

### E12. Reorganize String (LC 767)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Rearrange string so no two adjacent characters are the same.

**Approach:** Max-heap by frequency. Alternately place the most frequent remaining char.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def reorganizeString(s):
    freq = Counter(s)
    heap = [(-cnt, ch) for ch, cnt in freq.items()]
    heapq.heapify(heap)
    result = []
    prev_cnt, prev_ch = 0, ''
    while heap:
        cnt, ch = heapq.heappop(heap)
        result.append(ch)
        if prev_cnt < 0:
            heapq.heappush(heap, (prev_cnt, prev_ch))
        prev_cnt, prev_ch = cnt + 1, ch
    res = "".join(result)
    return res if len(res) == len(s) else ""

assert reorganizeString("aab") in ["aba"]
assert reorganizeString("aaab") == ""
assert len(reorganizeString("aabb")) == 4
print("All tests passed!")

### E13. Top K Frequent Words (LC 692)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Return the k most frequent words; ties broken alphabetically.

**Approach:** Count frequencies; use min-heap of size k with (-freq, word) as key.
**Time:** O(n log k) | **Space:** O(n)

In [ ]:
def topKFrequent(words, k):
    freq = Counter(words)
    # heapq.nlargest with tuple key: higher freq first, then alphabetical
    return heapq.nlargest(k, freq.keys(), key=lambda w: (freq[w], [-ord(c) for c in w]))

assert topKFrequent(["i","love","leetcode","i","love","coding"],2) == ["i","love"]
assert topKFrequent(["the","day","is","sunny","the","the","the","sunny","is","is"],4) == ["the","is","sunny","day"]
print("All tests passed!")

### E14. Minimum Cost to Connect Sticks (LC 1167)

> 🏢 **Asked by:** Amazon, Google
Connect all sticks; cost = sum of two sticks connected. Minimize total cost.

**Approach:** Greedy: always merge the two shortest sticks (Huffman coding). Use min-heap.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def connectSticks(sticks):
    heapq.heapify(sticks)
    total = 0
    while len(sticks) > 1:
        a = heapq.heappop(sticks)
        b = heapq.heappop(sticks)
        total += a + b
        heapq.heappush(sticks, a + b)
    return total

assert connectSticks([2,4,3]) == 14
assert connectSticks([1,8,3,5]) == 30
assert connectSticks([5]) == 0
print("All tests passed!")

### E15. The K Weakest Rows in a Matrix (LC 1337)

> 🏢 **Asked by:** Amazon, Google
Return indices of k weakest rows (fewest 1s; ties by row index).

**Approach:** Count 1s per row (binary search for efficiency); nsmallest k by (count, index).
**Time:** O(m log n + m log k) | **Space:** O(m)

In [ ]:
import bisect

def kWeakestRows(mat, k):
    strengths = [(sum(row), i) for i, row in enumerate(mat)]
    return [i for _, i in heapq.nsmallest(k, strengths)]

assert kWeakestRows([[1,1,0,0,0],[1,1,1,1,0],[1,0,0,0,0],[1,1,0,0,0],[1,1,1,1,1]], 3) == [2,0,3]
assert kWeakestRows([[1,0,0,0],[1,1,1,1],[1,0,0,0],[1,0,0,0]], 2) == [0,2]
print("All tests passed!")

### E16. Furthest Building You Can Reach (LC 1642)

> 🏢 **Asked by:** Amazon, Google
Using bricks and ladders, find the furthest building you can reach.

**Approach:** Use ladders for largest climbs. Min-heap of size `ladders` tracks ladder-used climbs. When exceeded, replace smallest ladder-use with bricks.
**Time:** O(n log L) | **Space:** O(L)

In [ ]:
def furthestBuilding(heights, bricks, ladders):
    heap = []  # min-heap of climbs where we used a ladder
    for i in range(len(heights)-1):
        diff = heights[i+1] - heights[i]
        if diff <= 0: continue
        heapq.heappush(heap, diff)
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
        if bricks < 0:
            return i
    return len(heights) - 1

assert furthestBuilding([4,2,7,6,9,14,12], 5, 1) == 4
assert furthestBuilding([4,12,2,7,3,18,20,3,19], 10, 2) == 7
assert furthestBuilding([14,3,19,3], 17, 0) == 3
print("All tests passed!")

### E17. Maximum Units on a Truck (LC 1710)

> 🏢 **Asked by:** Amazon, Google
Load a truck with at most truckSize boxes; maximize total units.

**Approach:** Greedy: sort box types by units per box (descending); fill greedily.
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def maximumUnits(boxTypes, truckSize):
    boxTypes.sort(key=lambda x: -x[1])
    total = 0
    for boxes, units in boxTypes:
        take = min(boxes, truckSize)
        total += take * units
        truckSize -= take
        if truckSize == 0: break
    return total

assert maximumUnits([[1,3],[2,2],[3,1]], 4) == 8
assert maximumUnits([[5,10],[2,5],[4,7],[3,9]], 10) == 91
print("All tests passed!")

### E18. Sort Array by Frequency (LC 1636)

> 🏢 **Asked by:** Amazon, Google
Sort array elements by increasing frequency; ties broken by decreasing value.

**Approach:** Count frequencies; sort with key=(freq, -val).
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def frequencySort2(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))

assert frequencySort2([1,1,2,2,2,3]) == [3,1,1,2,2,2]
assert frequencySort2([2,3,1,3,2]) == [1,3,3,2,2]
print("All tests passed!")

### E19. Find K Pairs with Smallest Sums (LC 373) — Easy Version

> 🏢 **Asked by:** Amazon, Google, Microsoft
Given two sorted arrays, find k pairs (u,v) with smallest sum (one from each array).

**Approach:** Min-heap seeded with (nums1[i]+nums2[0], i, 0). Expand by incrementing j.
**Time:** O(k log k) | **Space:** O(k)

In [ ]:
def kSmallestPairs(nums1, nums2, k):
    if not nums1 or not nums2: return []
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
    heapq.heapify(heap)
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):
            heapq.heappush(heap, (nums1[i] + nums2[j+1], i, j+1))
    return result

assert kSmallestPairs([1,7,11],[2,4,6],3) == [[1,2],[1,4],[1,6]]
assert kSmallestPairs([1,1,2],[1,2,3],2) == [[1,1],[1,1]]
print("All tests passed!")

### E20. Find Kth Smallest/Largest Using Heap

> 🏢 **Asked by:** Amazon, Google
Given an unsorted array, find the kth largest element using a heap.

**Approach:** Min-heap of size k; root = kth largest.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def findKthLargest(nums, k):
    heap = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]

assert findKthLargest([3,2,1,5,6,4], 2) == 5
assert findKthLargest([3,2,3,1,2,4,5,5,6], 4) == 4
assert findKthLargest([1], 1) == 1
print("All tests passed!")

---
## Medium Problems (15)

### M1. Top K Frequent Elements (LC 347)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Return the k most frequent elements.

**Approach:** Counter + heapq.nlargest by frequency.
**Time:** O(n log k) | **Space:** O(n)

In [ ]:
def topKFrequentElements(nums, k):
    freq = Counter(nums)
    return heapq.nlargest(k, freq.keys(), key=freq.get)

assert set(topKFrequentElements([1,1,1,2,2,3], 2)) == {1, 2}
assert topKFrequentElements([1], 1) == [1]
print("All tests passed!")

### M2. Kth Largest Element in an Array (LC 215)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Find the kth largest element without full sorting.

**Approach:** Min-heap of size k. O(n log k) time, O(k) space.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def findKthLargestFull(nums, k):
    heap = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]

assert findKthLargestFull([3,2,1,5,6,4], 2) == 5
assert findKthLargestFull([3,2,3,1,2,4,5,5,6], 4) == 4
print("All tests passed!")

### M3. Task Scheduler (LC 621)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Given tasks and cooldown n, find minimum intervals to complete all tasks.

**Approach:** Greedily schedule most frequent tasks first. Use max-heap + cooldown queue.
**Time:** O(n) | **Space:** O(1) (26 letters)

In [ ]:
import collections

def leastInterval(tasks, n):
    freq = Counter(tasks)
    heap = [-f for f in freq.values()]
    heapq.heapify(heap)
    time = 0
    queue = collections.deque()  # (count, available_time)
    while heap or queue:
        time += 1
        if heap:
            cnt = heapq.heappop(heap) + 1  # decrement count
            if cnt < 0:
                queue.append((cnt, time + n))
        if queue and queue[0][1] == time:
            heapq.heappush(heap, queue.popleft()[0])
    return time

assert leastInterval(["A","A","A","B","B","B"], 2) == 8
assert leastInterval(["A","C","A","B","D","B"], 1) == 6
assert leastInterval(["A","A","A","B","B","B"], 0) == 6
print("All tests passed!")

### M4. Reorganize String Full (LC 767)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Rearrange string so no two adjacent characters are the same (full version).

**Approach:** Max-heap by frequency; always place most frequent char, then second most frequent.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def reorganizeStringFull(s):
    freq = Counter(s)
    if max(freq.values()) > (len(s) + 1) // 2:
        return ""
    heap = [(-cnt, ch) for ch, cnt in freq.items()]
    heapq.heapify(heap)
    result = []
    prev_cnt, prev_ch = 0, ''
    while heap:
        cnt, ch = heapq.heappop(heap)
        result.append(ch)
        if prev_cnt < 0:
            heapq.heappush(heap, (prev_cnt, prev_ch))
        prev_cnt, prev_ch = cnt + 1, ch
    return "".join(result)

res = reorganizeStringFull("aab")
assert res and res[0] != res[1]
assert reorganizeStringFull("aaab") == ""
res2 = reorganizeStringFull("aabb")
assert all(res2[i] != res2[i+1] for i in range(len(res2)-1))
print("All tests passed!")

### M5. Find K Pairs with Smallest Sums Full (LC 373)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Full solution for k smallest sum pairs from two sorted arrays.

**Approach:** Min-heap with (sum, i, j). For each popped (i,j), push (i, j+1).
**Time:** O(k log k) | **Space:** O(k)

In [ ]:
def kSmallestPairsFull(nums1, nums2, k):
    if not nums1 or not nums2: return []
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
    heapq.heapify(heap)
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):
            heapq.heappush(heap, (nums1[i] + nums2[j+1], i, j+1))
    return result

assert kSmallestPairsFull([1,7,11],[2,4,6],3) == [[1,2],[1,4],[1,6]]
assert len(kSmallestPairsFull([1,2],[3],3)) == 2
print("All tests passed!")

### M6. Kth Smallest Element in a Sorted Matrix (LC 378)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Find kth smallest element in an n×n matrix where rows and columns are sorted.

**Approach:** Min-heap seeded with first column. Pop k times, push right neighbor each time.
**Time:** O(k log n) | **Space:** O(n)

In [ ]:
def kthSmallest(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(n)]
    heapq.heapify(heap)
    for _ in range(k):
        val, r, c = heapq.heappop(heap)
        if c + 1 < n:
            heapq.heappush(heap, (matrix[r][c+1], r, c+1))
    return val

assert kthSmallest([[1,5,9],[10,11,13],[12,13,15]], 8) == 13
assert kthSmallest([[-5]], 1) == -5
print("All tests passed!")

### M7. IPO (LC 502)

> 🏢 **Asked by:** Amazon, Google, Meta
Maximize capital after at most k projects; each project has profit and min capital required.

**Approach:** Sort by capital. Min-heap by capital; max-heap for available profits. Unlock projects as capital grows.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def findMaximizedCapital(k, w, profits, capital):
    n = len(profits)
    projects = sorted(zip(capital, profits))
    available = []  # max-heap of profits
    i = 0
    for _ in range(k):
        # Unlock all projects we can afford
        while i < n and projects[i][0] <= w:
            heapq.heappush(available, -projects[i][1])
            i += 1
        if not available: break
        w += -heapq.heappop(available)
    return w

assert findMaximizedCapital(2, 0, [1,2,3], [0,1,1]) == 4
assert findMaximizedCapital(3, 0, [1,2,3], [0,1,2]) == 6
print("All tests passed!")

### M8. Maximum Subsequence Score (LC 2542)

> 🏢 **Asked by:** Amazon, Google
Choose k indices; score = (sum of nums1[i]) * min(nums2[i]).

**Approach:** Sort by nums2 descending. For each new minimum (nums2[i]), maintain top-k nums1 values in min-heap.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def maxScore(nums1, nums2, k):
    pairs = sorted(zip(nums1, nums2), key=lambda x: -x[1])
    heap = []
    cur_sum = 0
    best = 0
    for n1, n2 in pairs:
        heapq.heappush(heap, n1)
        cur_sum += n1
        if len(heap) > k:
            cur_sum -= heapq.heappop(heap)
        if len(heap) == k:
            best = max(best, cur_sum * n2)
    return best

assert maxScore([1,3,3,2],[2,1,3,4],3) == 12
assert maxScore([4,2,3,1,1],[7,5,10,9,6],1) == 30
print("All tests passed!")

### M9. Minimum Cost to Hire K Workers (LC 857)

> 🏢 **Asked by:** Amazon, Google, Meta
Hire exactly k workers; each has quality and wage. Pay proportional to quality.

**Approach:** Sort by wage/quality ratio. For each worker as the one with max ratio, take k workers with smallest quality. Max-heap of size k.
**Time:** O(n log n + n log k) | **Space:** O(k)

In [ ]:
def mincostToHireWorkers(quality, wage, k):
    workers = sorted((w/q, q) for w, q in zip(wage, quality))
    heap = []  # max-heap (negate quality)
    q_sum = 0
    best = float("inf")
    for ratio, q in workers:
        heapq.heappush(heap, -q)
        q_sum += q
        if len(heap) > k:
            q_sum += heapq.heappop(heap)  # remove largest quality
        if len(heap) == k:
            best = min(best, ratio * q_sum)
    return best

assert abs(mincostToHireWorkers([10,20,5],[70,50,30],2) - 105.0) < 1e-5
assert abs(mincostToHireWorkers([3,1,10,10,1],[4,8,2,2,7],3) - 30.666666666666664) < 1e-5
print("All tests passed!")

### M10. Meeting Rooms III (LC 2402)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given n meeting rooms and meetings, find the room that held the most meetings.

**Approach:** Two heaps: available rooms (min-heap by id) and busy rooms (min-heap by end_time, room_id).
**Time:** O(m log n) | **Space:** O(n)

In [ ]:
def mostBooked(n, meetings):
    meetings.sort()
    available = list(range(n))
    heapq.heapify(available)
    busy = []  # (end_time, room_id)
    count = [0] * n
    for start, end in meetings:
        # Free up rooms whose meetings have ended
        while busy and busy[0][0] <= start:
            _, room = heapq.heappop(busy)
            heapq.heappush(available, room)
        if available:
            room = heapq.heappop(available)
            heapq.heappush(busy, (end, room))
        else:
            end_time, room = heapq.heappop(busy)
            heapq.heappush(busy, (end_time + (end - start), room))
        count[room] += 1
    return count.index(max(count))

assert mostBooked(2, [[0,10],[1,5],[2,7],[3,4]]) == 0
assert mostBooked(3, [[1,20],[2,10],[3,5],[4,9],[6,8]]) == 1
print("All tests passed!")

### M11. Car Pooling with Heap (LC 1094)

> 🏢 **Asked by:** Amazon, Google, Lyft, Uber
Given trips and capacity, determine if all passengers can be picked up.

**Approach:** Sort trips by start; use min-heap of (end, passengers) to track active trips.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def carPooling(trips, capacity):
    trips.sort(key=lambda x: x[1])
    heap = []  # (end, passengers)
    cur = 0
    for passengers, start, end in trips:
        while heap and heap[0][0] <= start:
            cur -= heapq.heappop(heap)[1]
        cur += passengers
        if cur > capacity: return False
        heapq.heappush(heap, (end, passengers))
    return True

assert carPooling([[2,1,5],[3,3,7]], 4) == False
assert carPooling([[2,1,5],[3,3,7]], 5) == True
assert carPooling([[3,2,7],[3,7,9],[8,3,9]], 11) == True
print("All tests passed!")

### M12. Minimum Number of Refueling Stops (LC 871)

> 🏢 **Asked by:** Amazon, Google
Find minimum refueling stops to reach target; greedy with max-heap.

**Approach:** Greedily use largest available fuel when needed. Max-heap of passed station fuels.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def minRefuelStops(target, startFuel, stations):
    heap = []  # max-heap (negate fuel)
    fuel = startFuel
    stops = 0
    prev = 0
    for loc, cap in stations + [[target, 0]]:
        fuel -= (loc - prev)
        while fuel < 0 and heap:
            fuel += -heapq.heappop(heap)
            stops += 1
        if fuel < 0: return -1
        heapq.heappush(heap, -cap)
        prev = loc
    return stops

assert minRefuelStops(1, 1, []) == 0
assert minRefuelStops(100, 10, [[10,100]]) == 1
assert minRefuelStops(100, 10, [[10,60],[20,30],[30,30],[60,40]]) == 2
print("All tests passed!")

### M13. Smallest Range Covering Elements from K Lists (LC 632)

> 🏢 **Asked by:** Google, Amazon
Find the smallest range [a,b] such that at least one element from each list falls in [a,b].

**Approach:** Min-heap with one element from each list. Track max. Advance min pointer.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def smallestRange(nums):
    heap = [(row[0], i, 0) for i, row in enumerate(nums)]
    heapq.heapify(heap)
    cur_max = max(row[0] for row in nums)
    best = [float("-inf"), float("inf")]
    while heap:
        cur_min, i, j = heapq.heappop(heap)
        if cur_max - cur_min < best[1] - best[0]:
            best = [cur_min, cur_max]
        if j + 1 == len(nums[i]): break
        nxt = nums[i][j+1]
        cur_max = max(cur_max, nxt)
        heapq.heappush(heap, (nxt, i, j+1))
    return best

assert smallestRange([[4,10,15,24,26],[0,9,12,20],[5,18,22,30]]) == [20,24]
assert smallestRange([[1,2,3],[1,2,3],[1,2,3]]) == [1,1]
print("All tests passed!")

### M14. Find Median from Data Stream (Partial) (LC 295)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Find the median of a stream of numbers (simplified version with sorted insert).

**Approach:** Two heaps: max-heap for lower half, min-heap for upper half. Balance sizes.
**Time:** O(log n) per add, O(1) find median | **Space:** O(n)

In [ ]:
class MedianFinder:
    def __init__(self):
        self.lo = []  # max-heap (lower half, negated)
        self.hi = []  # min-heap (upper half)

    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2.0
print("All tests passed!")

### M15. Merge K Sorted Lists (LC 23) — Heap Approach

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Merge k sorted linked lists into one sorted list.

**Approach:** Min-heap of (value, list_index, node). Push first node of each list; pop and push next.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def mergeKLists(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = ListNode(0)
    cur = dummy
    while heap:
        val, i, node = heapq.heappop(heap)
        cur.next = node
        cur = cur.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

def to_list(node):
    res = []
    while node: res.append(node.val); node = node.next
    return res

def from_list(lst):
    if not lst: return None
    head = ListNode(lst[0])
    cur = head
    for v in lst[1:]: cur.next = ListNode(v); cur = cur.next
    return head

l1 = from_list([1,4,5])
l2 = from_list([1,3,4])
l3 = from_list([2,6])
assert to_list(mergeKLists([l1,l2,l3])) == [1,1,2,3,4,4,5,6]
assert mergeKLists([]) is None
print("All tests passed!")

---
## Hard Problems (10)

### H1. Find Median from Data Stream (LC 295)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Full implementation with both addNum and findMedian operations.

**Approach:** Two heaps: max-heap (lo, lower half) and min-heap (hi, upper half). Maintain size balance.
**Time:** O(log n) addNum, O(1) findMedian | **Space:** O(n)

In [ ]:
class MedianFinderFull:
    def __init__(self):
        self.lo = []  # max-heap (negated)
        self.hi = []  # min-heap

    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

mf = MedianFinderFull()
for n in [5, 3, 8, 1, 9, 2, 7]:
    mf.addNum(n)
assert mf.findMedian() == 5.0
mf2 = MedianFinderFull()
mf2.addNum(1); mf2.addNum(2)
assert mf2.findMedian() == 1.5
mf2.addNum(3)
assert mf2.findMedian() == 2.0
print("All tests passed!")

### H2. Sliding Window Median (LC 480)

> 🏢 **Asked by:** Amazon, Google, Meta
Find the median of each sliding window of size k.

**Approach:** Two heaps (lo max-heap, hi min-heap) + lazy deletion dict. Slide window maintaining balance.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
import sortedcontainers

def medianSlidingWindow(nums, k):
    """Sliding window median using SortedList for O(n log k) time."""
    sl = sortedcontainers.SortedList()
    result = []
    for i, n in enumerate(nums):
        sl.add(n)
        if i >= k:
            sl.remove(nums[i - k])
        if i >= k - 1:
            if k % 2 == 1:
                result.append(float(sl[k // 2]))
            else:
                result.append((sl[k // 2 - 1] + sl[k // 2]) / 2.0)
    return result

assert medianSlidingWindow([1,3,-1,-3,5,3,6,7], 3) == [1.0,-1.0,-1.0,3.0,5.0,6.0]
assert medianSlidingWindow([1,2], 1) == [1.0,2.0]
print("All tests passed!")

### H3. IPO Full (LC 502)

> 🏢 **Asked by:** Amazon, Google, Meta
Full greedy capital maximization with k projects.

**Approach:** Sort by capital. Use max-heap for available profits. Greedily pick most profitable affordable project each round.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def findMaximizedCapitalFull(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    available = []
    i = 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(available, -projects[i][1])
            i += 1
        if not available: break
        w += -heapq.heappop(available)
    return w

assert findMaximizedCapitalFull(2, 0, [1,2,3], [0,1,1]) == 4
assert findMaximizedCapitalFull(3, 0, [1,2,3], [0,1,2]) == 6
assert findMaximizedCapitalFull(1, 0, [1,2,3], [1,1,2]) == 0
print("All tests passed!")

### H4. Minimum Cost to Hire K Workers Full (LC 857)

> 🏢 **Asked by:** Amazon, Google, Meta
Full implementation with edge cases.

**Approach:** Sort workers by wage/quality ratio. Slide over ratios; maintain top-k min quality sum using max-heap.
**Time:** O(n log n + n log k) | **Space:** O(k)

In [ ]:
def mincostToHireWorkersFull(quality, wage, k):
    n = len(quality)
    workers = sorted((wage[i]/quality[i], quality[i]) for i in range(n))
    heap = []  # max-heap of quality (negated)
    q_sum = 0
    result = float("inf")
    for ratio, q in workers:
        heapq.heappush(heap, -q)
        q_sum += q
        if len(heap) > k:
            q_sum += heapq.heappop(heap)
        if len(heap) == k:
            result = min(result, ratio * q_sum)
    return result

assert abs(mincostToHireWorkersFull([10,20,5],[70,50,30],2) - 105.0) < 1e-5
assert abs(mincostToHireWorkersFull([3,1,10,10,1],[4,8,2,2,7],3) - 30.666666666666664) < 1e-5
print("All tests passed!")

### H5. The Skyline Problem (LC 218)

> 🏢 **Asked by:** Google, Amazon, Microsoft, Bloomberg
Return the skyline outline as a list of [x, height] key points.

**Approach:** Event sweep: building start adds height, end removes it. Use max-heap or sorted container.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from sortedcontainers import SortedList

def getSkyline(buildings):
    # Sweep left to right. A start event adds a height to the active multiset,
    # an end event removes it; a key point is emitted whenever the tallest
    # active building changes. Negating the height on start events makes taller
    # buildings sort first at the same x, so a start never emits a lower point.
    events = []
    for left, right, h in buildings:
        events.append((left, -h))   # start
        events.append((right, h))   # end
    events.sort()

    active = SortedList([0])        # 0 is the ground, so the set is never empty
    result = []
    for x, h in events:
        if h < 0:
            active.add(-h)
        else:
            active.remove(h)        # O(log n) -- the reason a plain heap needs
        tallest = active[-1]        # lazy deletion instead
        if not result or result[-1][1] != tallest:
            result.append([x, tallest])
    return result

def getSkylineHeap(buildings):
    # Same sweep with a max-heap and lazy deletion: heapq cannot remove an
    # arbitrary element, so ended buildings are counted and popped once they
    # reach the top.
    events = []
    for left, right, h in buildings:
        events.append((left, -h))
        events.append((right, h))
    events.sort()

    heap = [0]
    ended = defaultdict(int)
    result = []
    for x, h in events:
        if h < 0:
            heapq.heappush(heap, h)
        else:
            ended[h] += 1
        while heap and ended[-heap[0]] > 0:
            ended[-heap[0]] -= 1
            heapq.heappop(heap)
        tallest = -heap[0]
        if not result or result[-1][1] != tallest:
            result.append([x, tallest])
    return result

expected = [[2,10],[3,15],[7,12],[12,0],[15,10],[20,8],[24,0]]
buildings = [[2,9,10],[3,7,15],[5,12,12],[15,20,10],[19,24,8]]
assert getSkyline(buildings) == expected
assert getSkylineHeap(buildings) == expected
assert getSkyline([[0,2,3],[2,5,3]]) == [[0,3],[5,0]]      # touching, same height
assert getSkyline([[1,2,1],[1,2,2],[1,2,3]]) == [[1,3],[2,0]]  # nested, same span
assert getSkyline([]) == []
assert getSkyline([[0,1,5]]) == [[0,5],[1,0]]
print("All tests passed!")

### H6. Maximum Performance of a Team (LC 1383)

> 🏢 **Asked by:** Google, Amazon
Pick at most k engineers; score = sum(speed) * min(efficiency).

**Approach:** Sort by efficiency descending. For each engineer as min-efficiency, maintain top-k speeds in min-heap.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def maxPerformance(n, speed, efficiency, k):
    MOD = 10**9 + 7
    engineers = sorted(zip(efficiency, speed), reverse=True)
    heap = []
    speed_sum = 0
    best = 0
    for eff, spd in engineers:
        heapq.heappush(heap, spd)
        speed_sum += spd
        if len(heap) > k:
            speed_sum -= heapq.heappop(heap)
        best = max(best, speed_sum * eff)
    return best % MOD

assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 2) == 60
assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 3) == 68
assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 4) == 72
print("All tests passed!")

### H7. Smallest Range Covering Elements from K Lists Full (LC 632)

> 🏢 **Asked by:** Google, Amazon
Full implementation handling edge cases.

**Approach:** Min-heap with (value, list_i, element_j). Track running max. Advance min; check if range shrinks.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def smallestRangeFull(nums):
    heap = [(row[0], i, 0) for i, row in enumerate(nums)]
    heapq.heapify(heap)
    cur_max = max(row[0] for row in nums)
    best_lo, best_hi = float("-inf"), float("inf")
    while len(heap) == len(nums):
        cur_min, i, j = heapq.heappop(heap)
        if cur_max - cur_min < best_hi - best_lo:
            best_lo, best_hi = cur_min, cur_max
        if j + 1 < len(nums[i]):
            nxt = nums[i][j+1]
            cur_max = max(cur_max, nxt)
            heapq.heappush(heap, (nxt, i, j+1))
        else:
            break
    return [best_lo, best_hi]

assert smallestRangeFull([[4,10,15,24,26],[0,9,12,20],[5,18,22,30]]) == [20,24]
assert smallestRangeFull([[1,2,3],[1,2,3],[1,2,3]]) == [1,1]
print("All tests passed!")

### H8. Merge K Sorted Lists Full (LC 23)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Full robust implementation of merge k sorted lists.

**Approach:** Min-heap storing (node.val, index, node). Index breaks tie for non-comparable nodes.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
def mergeKListsFull(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = ListNode(0)
    cur = dummy
    while heap:
        val, i, node = heapq.heappop(heap)
        cur.next = node
        cur = cur.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

l1 = from_list([1,4,5])
l2 = from_list([1,3,4])
l3 = from_list([2,6])
assert to_list(mergeKListsFull([l1,l2,l3])) == [1,1,2,3,4,4,5,6]
assert mergeKListsFull([]) is None
assert to_list(mergeKListsFull([from_list([])])) == []
print("All tests passed!")

### H9. Minimum Number of Refueling Stops Full (LC 871)

> 🏢 **Asked by:** Amazon, Google
Full solution with all edge cases.

**Approach:** Greedy max-heap: collect all stations passed; when out of fuel, refuel from largest available station.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def minRefuelStopsFull(target, startFuel, stations):
    heap = []  # max-heap (negate fuel)
    fuel = startFuel
    stops = 0
    prev = 0
    for loc, cap in stations + [[target, 0]]:
        fuel -= (loc - prev)
        while fuel < 0 and heap:
            fuel += -heapq.heappop(heap)
            stops += 1
        if fuel < 0:
            return -1
        heapq.heappush(heap, -cap)
        prev = loc
    return stops

assert minRefuelStopsFull(1, 1, []) == 0
assert minRefuelStopsFull(100, 10, [[10,100]]) == 1
assert minRefuelStopsFull(100, 10, [[10,60],[20,30],[30,30],[60,40]]) == 2
assert minRefuelStopsFull(100, 10, [[10,60],[20,30],[30,30],[60,40],[90,60]]) == 2
print("All tests passed!")

### H10. Find the Kth Smallest Sum of a Matrix With Sorted Rows (LC 1439)

> 🏢 **Asked by:** Amazon, Google
Return the kth smallest sum where one element is chosen from each row.

**Approach:** Merge rows iteratively. After each merge, keep only k smallest sums using a heap.
**Time:** O(m * k log k) | **Space:** O(k)

In [ ]:
def kthSmallest_matrix(mat, k):
    # Merge two sorted arrays keeping only k smallest sums
    def merge(a, b, k):
        heap = [(a[0]+b[0], 0, 0)]
        visited = {(0,0)}
        result = []
        while heap and len(result) < k:
            val, i, j = heapq.heappop(heap)
            result.append(val)
            if i+1 < len(a) and (i+1,j) not in visited:
                heapq.heappush(heap, (a[i+1]+b[j], i+1, j))
                visited.add((i+1,j))
            if j+1 < len(b) and (i,j+1) not in visited:
                heapq.heappush(heap, (a[i]+b[j+1], i, j+1))
                visited.add((i,j+1))
        return result

    cur = sorted(mat[0])[:k]
    for row in mat[1:]:
        cur = merge(cur, sorted(row), k)
    return cur[k-1]

assert kthSmallest_matrix([[1,3,11],[2,4,6]], 5) == 7
assert kthSmallest_matrix([[1,3,11],[2,4,6]], 9) == 17
assert kthSmallest_matrix([[1,10,10],[1,4,5],[2,3,6]], 7) == 9
print("All tests passed!")

## Easy Problems (21-40)

### E21. Kth Smallest Element in Sorted Matrix – Heap Warmup (LC 378)

> 🏢 **Asked by:** Amazon, Google
Given an n×n matrix where each row and column is sorted, return the kth smallest element.

**Approach:** Push (val, r, c) into a min-heap starting with the first column. Pop k-1 times, expanding right and down.
**Time:** O(k log k) | **Space:** O(k)

In [ ]:
import heapq

def kthSmallestMatrix(matrix, k):
    n = len(matrix)
    heap = [(matrix[0][0], 0, 0)]
    visited = {(0, 0)}
    for _ in range(k - 1):
        val, r, c = heapq.heappop(heap)
        for nr, nc in [(r+1, c), (r, c+1)]:
            if nr < n and nc < n and (nr, nc) not in visited:
                heapq.heappush(heap, (matrix[nr][nc], nr, nc))
                visited.add((nr, nc))
    return heapq.heappop(heap)[0]

assert kthSmallestMatrix([[1,5,9],[10,11,13],[12,13,15]], 8) == 13
assert kthSmallestMatrix([[1,2],[1,3]], 2) == 1
assert kthSmallestMatrix([[-5]], 1) == -5
print("All tests passed!")

### E22. Find Median from Running Stream – Step by Step (LC 295 simplified)

> 🏢 **Asked by:** Amazon, Google
Given a list of numbers arriving one by one, print the running median after each insertion.

**Approach:** Use a max-heap for the lower half and a min-heap for the upper half. Balance sizes so they differ by at most 1.
**Time:** O(log n) per insertion | **Space:** O(n)

In [ ]:
import heapq

def running_medians(stream):
    lo = []  # max-heap (negate)
    hi = []  # min-heap
    medians = []
    for num in stream:
        heapq.heappush(lo, -num)
        heapq.heappush(hi, -heapq.heappop(lo))
        if len(hi) > len(lo):
            heapq.heappush(lo, -heapq.heappop(hi))
        if len(lo) == len(hi):
            medians.append((-lo[0] + hi[0]) / 2)
        else:
            medians.append(-lo[0])
    return medians

assert running_medians([1]) == [1]
assert running_medians([1, 2]) == [1, 1.5]
assert running_medians([5, 3, 8]) == [5, 4.0, 5]
print("All tests passed!")

### E23. Sort Characters By Frequency (LC 451)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Sort characters in a string by descending frequency.

**Approach:** Count frequencies with a Counter, push (-freq, char) into a max-heap, then build result.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq
from collections import Counter

def frequencySort(s):
    freq = Counter(s)
    heap = [(-cnt, ch) for ch, cnt in freq.items()]
    heapq.heapify(heap)
    result = []
    while heap:
        cnt, ch = heapq.heappop(heap)
        result.append(ch * (-cnt))
    return ''.join(result)

assert frequencySort('tree') in ('eert', 'eetr')
assert frequencySort('cccaaa') in ('cccaaa', 'aaaccc')
assert frequencySort('Aabb') in ('bbaA', 'bbAa')
print("All tests passed!")

### E24. Relative Sort Array (LC 1122)

> 🏢 **Asked by:** Amazon, Google
Sort arr1 so that elements that appear in arr2 come first (in arr2's order), then remaining in ascending order.

**Approach:** Use a rank map from arr2; sort arr1 with key (rank if present else (len(arr2), val)).
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def relativeSortArray(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: (rank[x], 0) if x in rank else (len(arr2), x))

assert relativeSortArray([2,3,1,3,2,4,6,7,9,2,19], [2,1,4,3,9,6]) == [2,2,2,1,4,3,3,9,6,7,19]
assert relativeSortArray([28,6,22,8,44,17], [22,28,8,6]) == [22,28,8,6,17,44]
print("All tests passed!")

### E25. Maximum Bags With Full Capacity of Rocks (LC 2279)

> 🏢 **Asked by:** Amazon, Google
Given bags with capacity[i] and rocks[i] already inside, and additionalRocks to distribute, maximize full bags.

**Approach:** Sort bags by remaining space. Greedily fill bags from smallest remaining space first.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def maximumBags(capacity, rocks, additionalRocks):
    space = sorted(c - r for c, r in zip(capacity, rocks))
    count = 0
    for s in space:
        if additionalRocks >= s:
            additionalRocks -= s
            count += 1
        else:
            break
    return count

assert maximumBags([2,3,4,5], [1,2,4,4], 2) == 3
assert maximumBags([10,2,2], [2,2,0], 100) == 3
assert maximumBags([5], [3], 1) == 0
print("All tests passed!")

### E26. Minimum Cost to Move Chips to Same Position (LC 1217)

> 🏢 **Asked by:** Amazon, Google
Move all chips to one position. Moving 2 steps costs 0; moving 1 step costs 1. Minimize cost.

**Approach:** Chips at odd positions must pay 1 each to reach any even position (and vice versa). Answer = min(odd_count, even_count).
**Time:** O(n) | **Space:** O(1)

In [ ]:
def minCostChips(position):
    odd = sum(1 for p in position if p % 2 == 1)
    even = len(position) - odd
    return min(odd, even)

assert minCostChips([1,2,3]) == 1
assert minCostChips([2,2,2,3,3]) == 2
assert minCostChips([1,1000000000]) == 1
print("All tests passed!")

### E27. Minimum Number of Moves to Seat Everyone (LC 2037)

> 🏢 **Asked by:** Amazon, Google
Move students to seats. Each move shifts a student by 1. Minimize total moves.

**Approach:** Sort both arrays and pair them by index. Sum of absolute differences.
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def minMovesToSeat(seats, students):
    return sum(abs(s - t) for s, t in zip(sorted(seats), sorted(students)))

assert minMovesToSeat([3,1,5], [2,7,4]) == 4
assert minMovesToSeat([4,1,5,9], [1,3,2,6]) == 7
assert minMovesToSeat([2,2,6,6], [1,3,2,6]) == 4
print("All tests passed!")

### E28. Find the Kth Largest Sum of a Subarray (small n, brute force + heap)

> 🏢 **Asked by:** Amazon, Google
Given a small array, find the kth largest subarray sum using brute force enumeration and a min-heap.

**Approach:** Enumerate all O(n²) subarrays, keep a min-heap of size k. The root is the answer.
**Time:** O(n² log k) | **Space:** O(k)

In [ ]:
import heapq

def kthLargestSubarraySum(nums, k):
    heap = []  # min-heap of size k
    n = len(nums)
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            heapq.heappush(heap, s)
            if len(heap) > k:
                heapq.heappop(heap)
    return heap[0]

assert kthLargestSubarraySum([3, -1, 4, 1], 3) == 5
assert kthLargestSubarraySum([1, 2, 3], 2) == 5
assert kthLargestSubarraySum([-1, -2, -3], 1) == -1
print("All tests passed!")

### E29. Least Number of Unique Integers After K Removals (LC 1481)

> 🏢 **Asked by:** Amazon, Google
Remove k elements to minimize the number of unique integers remaining.

**Approach:** Count frequencies, sort by frequency ascending. Remove least frequent elements first.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from collections import Counter

def findLeastNumOfUniqueInts(arr, k):
    freq = sorted(Counter(arr).values())
    unique = len(freq)
    for f in freq:
        if k >= f:
            k -= f
            unique -= 1
        else:
            break
    return unique

assert findLeastNumOfUniqueInts([5,5,4], 1) == 1
assert findLeastNumOfUniqueInts([4,3,1,1,3,3,2], 3) == 2
assert findLeastNumOfUniqueInts([1], 1) == 0
print("All tests passed!")

### E30. Minimum Number of Pushes to Type Word (LC 3014 simplified)

> 🏢 **Asked by:** Amazon, Google
Map 26 letters to 8 keys (2-9). The ith letter mapped to a key requires ceil(i/8) pushes. Minimize total pushes.

**Approach:** Sort letter frequencies descending. Assign most frequent letters first (1 push each), then 2 pushes, etc.
**Time:** O(n + 26 log 26) | **Space:** O(26)

In [ ]:
from collections import Counter

def minimumPushes(word):
    freq = sorted(Counter(word).values(), reverse=True)
    total = 0
    for i, f in enumerate(freq):
        total += f * (i // 8 + 1)
    return total

assert minimumPushes('abcde') == 5
assert minimumPushes('xyzxyzxyzxyz') == 12
assert minimumPushes('aabbccddeeffgghhiiiiii') == 24
print("All tests passed!")

### E31. Sort Array by Parity (LC 905)

> 🏢 **Asked by:** Amazon, Google
Return array with all even integers before all odd integers.

**Approach:** Two-pointer or partition sort. Even elements go to the front.
**Time:** O(n) | **Space:** O(1)

In [ ]:
def sortArrayByParity(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 > nums[r] % 2:
            nums[l], nums[r] = nums[r], nums[l]
        if nums[l] % 2 == 0: l += 1
        if nums[r] % 2 == 1: r -= 1
    return nums

res = sortArrayByParity([3,1,2,4])
assert all(res[i] % 2 == 0 for i in range(res.index(next(x for x in res if x%2!=0))))
assert sortArrayByParity([0]) == [0]
assert sortArrayByParity([2,4,6]) == [2,4,6]
print("All tests passed!")

### E32. Sort Array by Parity II (LC 922)

> 🏢 **Asked by:** Amazon, Google
Rearrange so nums[i] is even if i is even, and odd if i is odd.

**Approach:** Two pointers: one for even indices, one for odd indices. Swap when they're both wrong.
**Time:** O(n) | **Space:** O(1)

In [ ]:
def sortArrayByParityII(nums):
    n = len(nums)
    j = 1  # pointer for odd index
    for i in range(0, n, 2):
        if nums[i] % 2 == 1:  # even index has odd number
            while nums[j] % 2 == 1:
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

res = sortArrayByParityII([4,2,5,7])
assert all(res[i] % 2 == i % 2 for i in range(len(res)))
res2 = sortArrayByParityII([2,3])
assert all(res2[i] % 2 == i % 2 for i in range(len(res2)))
print("All tests passed!")

### E33. Rank Teams by Votes (LC 1366)

> 🏢 **Asked by:** Amazon, Google
Rank teams based on votes. Each voter ranks teams; final rank is lexicographic on vote vectors.

**Approach:** For each team, count how many 1st-place votes, 2nd-place votes, etc. Sort by this vector descending, then by team name.
**Time:** O(m*n + n log n) | **Space:** O(n*m)

In [ ]:
def rankTeams(votes):
    n = len(votes[0])
    score = {t: [0]*n for t in votes[0]}
    for vote in votes:
        for rank, team in enumerate(vote):
            score[team][rank] += 1
    return ''.join(sorted(votes[0], key=lambda t: (score[t], [-ord(t)]), reverse=True))

assert rankTeams(['ABC','ACB','ABC','ACB','ACB']) == 'ACB'
assert rankTeams(['WXYZ','XYZW']) == 'XWYZ'
assert rankTeams(['ZMNAGUEDSJYLBOPHRQICWFXTVK']) == 'ZMNAGUEDSJYLBOPHRQICWFXTVK'
print("All tests passed!")

### E34. Maximum Number of Coins You Can Get (LC 1561)

> 🏢 **Asked by:** Amazon, Google
3n piles of coins; you, opponent, and a 'giveaway' each take a pile per turn. Maximize your coins.

**Approach:** Sort descending. You always take 2nd largest in each triplet (skip 1st to opponent, skip last n to giveaway).
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def maxCoins(piles):
    piles.sort()
    n = len(piles) // 3
    total = 0
    i = len(piles) - 2  # second from end in each triplet
    for _ in range(n):
        total += piles[i]
        i -= 2
    return total

assert maxCoins([2,4,1,2,7,8]) == 9
assert maxCoins([2,4,5]) == 4
assert maxCoins([9,8,7,6,5,1,2,3,4]) == 18
print("All tests passed!")

### E35. Largest Perimeter Triangle (LC 976)

> 🏢 **Asked by:** Amazon, Google
Given side lengths, find the largest perimeter of a valid triangle (sum of 2 sides > 3rd side).

**Approach:** Sort descending. Check consecutive triples; the first valid triple gives the largest perimeter.
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def largestPerimeter(nums):
    nums.sort(reverse=True)
    for i in range(len(nums) - 2):
        if nums[i] < nums[i+1] + nums[i+2]:
            return nums[i] + nums[i+1] + nums[i+2]
    return 0

assert largestPerimeter([2,1,2]) == 5
assert largestPerimeter([1,2,1,10]) == 0
assert largestPerimeter([3,6,2,3]) == 8
print("All tests passed!")

### E36. Minimum Subsequence in Non-Increasing Order (LC 1403)

> 🏢 **Asked by:** Amazon, Google
Find the minimum-length subsequence whose sum is greater than the rest of the array.

**Approach:** Sort descending, greedily pick from largest. Stop once picked_sum > total - picked_sum.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def minSubsequence(nums):
    nums.sort(reverse=True)
    total = sum(nums)
    picked = 0
    result = []
    for n in nums:
        picked += n
        result.append(n)
        if picked > total - picked:
            break
    return result

assert minSubsequence([4,3,10,9,8]) == [10,9]
assert minSubsequence([4,4,7,6,7]) == [7,7,6]
assert minSubsequence([6]) == [6]
print("All tests passed!")

### E37. How Many Numbers Are Smaller Than the Current Number (LC 1365)

> 🏢 **Asked by:** Amazon, Google
For each element, count how many numbers in the array are strictly smaller.

**Approach:** Sort array, use the index of the first occurrence as the count (via bisect_left).
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import bisect

def smallerNumbersThanCurrent(nums):
    sorted_nums = sorted(nums)
    return [bisect.bisect_left(sorted_nums, n) for n in nums]

assert smallerNumbersThanCurrent([8,1,2,2,3]) == [4,0,1,1,3]
assert smallerNumbersThanCurrent([6,5,4,8]) == [2,1,0,3]
assert smallerNumbersThanCurrent([7,7,7,7]) == [0,0,0,0]
print("All tests passed!")

### E38. Maximum Product of Three Numbers (LC 628)

> 🏢 **Asked by:** Amazon, Google
Find the maximum product of any three numbers in an array.

**Approach:** Sort. Max product is either top-3 or bottom-2 * top-1 (two negatives * one positive).
**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def maximumProduct(nums):
    nums.sort()
    return max(nums[-1]*nums[-2]*nums[-3], nums[0]*nums[1]*nums[-1])

assert maximumProduct([1,2,3]) == 6
assert maximumProduct([1,2,3,4]) == 24
assert maximumProduct([-1,-2,-3]) == -6
assert maximumProduct([-4,-3,-2,-1,60]) == 720
print("All tests passed!")

### E39. Third Largest Number (LC 414) – Heap Approach

> 🏢 **Asked by:** Amazon, Google
Return the third distinct maximum. If it doesn't exist, return the maximum.

**Approach:** Use a min-heap of size 3, storing distinct values. Pop result at the end.
**Time:** O(n) | **Space:** O(1)

In [ ]:
import heapq

def thirdMax(nums):
    heap = list(set(nums))
    heapq.heapify(heap)
    while len(heap) > 3:
        heapq.heappop(heap)
    if len(heap) < 3:
        return max(heap)
    return heap[0]

assert thirdMax([3,2,1]) == 1
assert thirdMax([1,2]) == 2
assert thirdMax([2,2,3,1]) == 1
print("All tests passed!")

### E40. Find N Unique Integers Sum up to Zero (LC 1304)

> 🏢 **Asked by:** Amazon, Google
Return an array of n unique integers that sum to zero.

**Approach:** Use 1, 2, ..., n-1 and negate their sum as the nth element.
**Time:** O(n) | **Space:** O(n)

In [ ]:
def sumZero(n):
    result = list(range(1, n))
    result.append(-sum(result))
    return result

for n in [1, 2, 3, 5]:
    res = sumZero(n)
    assert len(res) == n
    assert len(set(res)) == n
    assert sum(res) == 0
print("All tests passed!")

## Medium Problems (16-30)

### M16. Kth Largest Element in an Array – QuickSelect (LC 215)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Find the kth largest element using QuickSelect (average O(n)).

**Approach:** Partition around a pivot; recurse only into the side containing the kth element.
**Time:** O(n) average, O(n²) worst | **Space:** O(1)

In [ ]:
import random

def findKthLargest_qs(nums, k):
    target = len(nums) - k  # kth largest = target-th smallest (0-indexed)

    def quickselect(l, r):
        pivot_idx = random.randint(l, r)
        nums[pivot_idx], nums[r] = nums[r], nums[pivot_idx]
        pivot = nums[r]
        store = l
        for i in range(l, r):
            if nums[i] <= pivot:
                nums[store], nums[i] = nums[i], nums[store]
                store += 1
        nums[store], nums[r] = nums[r], nums[store]
        if store == target: return nums[store]
        elif store < target: return quickselect(store + 1, r)
        else: return quickselect(l, store - 1)

    return quickselect(0, len(nums) - 1)

assert findKthLargest_qs([3,2,1,5,6,4], 2) == 5
assert findKthLargest_qs([3,2,3,1,2,4,5,5,6], 4) == 4
print("All tests passed!")

### M17. K Closest Points to Origin – QuickSelect Version (LC 973)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Uber
Find k closest points using QuickSelect on squared distances.

**Approach:** Partition points by distance. After partitioning, left side has the k closest.
**Time:** O(n) average | **Space:** O(1)

In [ ]:
import random

def kClosest_qs(points, k):
    def dist(p): return p[0]**2 + p[1]**2

    def quickselect(l, r):
        if l >= r: return
        pivot_i = random.randint(l, r)
        points[pivot_i], points[r] = points[r], points[pivot_i]
        pivot_d = dist(points[r])
        store = l
        for i in range(l, r):
            if dist(points[i]) <= pivot_d:
                points[store], points[i] = points[i], points[store]
                store += 1
        points[store], points[r] = points[r], points[store]
        if store == k: return
        elif store < k: quickselect(store + 1, r)
        else: quickselect(l, store - 1)

    quickselect(0, len(points) - 1)
    return points[:k]

res = kClosest_qs([[1,3],[-2,2]], 1)
assert res == [[-2,2]] or dist_sq(res[0]) <= dist_sq([1,3])
def dist_sq(p): return p[0]**2+p[1]**2
res2 = kClosest_qs([[3,3],[5,-1],[-2,4]], 2)
assert all(dist_sq(p) <= dist_sq([5,-1]) for p in res2)
print("All tests passed!")

### M18. Find K-th Largest XOR Coordinate Value (LC 1738)

> 🏢 **Asked by:** Amazon, Google
Compute prefix XOR matrix, find kth largest value using a heap.

**Approach:** Build 2D prefix XOR array. Collect all values into a heap, return kth largest.
**Time:** O(m*n log k) | **Space:** O(m*n)

In [ ]:
import heapq

def kthLargestXOR(matrix, k):
    m, n = len(matrix), len(matrix[0])
    xor = [[0]*n for _ in range(m)]
    heap = []
    for i in range(m):
        for j in range(n):
            xor[i][j] = matrix[i][j]
            if i > 0: xor[i][j] ^= xor[i-1][j]
            if j > 0: xor[i][j] ^= xor[i][j-1]
            if i > 0 and j > 0: xor[i][j] ^= xor[i-1][j-1]
            heapq.heappush(heap, xor[i][j])
            if len(heap) > k:
                heapq.heappop(heap)
    return heap[0]

assert kthLargestXOR([[5,2],[1,6]], 1) == 7
assert kthLargestXOR([[5,2],[1,6]], 2) == 5
assert kthLargestXOR([[5,2],[1,6]], 3) == 4
assert kthLargestXOR([[5,2],[1,6]], 4) == 0
print("All tests passed!")

### M19. Maximum Average Pass Ratio (LC 1792)

> 🏢 **Asked by:** Amazon, Google
Distribute extra students to maximize average pass ratio using a max-heap on marginal gain.

**Approach:** Key insight: gain = (pass+1)/(total+1) - pass/total. Max-heap on gain, greedily assign each extra student to the class with highest gain.
**Time:** O((n + extraStudents) log n) | **Space:** O(n)

In [ ]:
import heapq

def maxAverageRatio(classes, extraStudents):
    def gain(p, t):
        return (p + 1) / (t + 1) - p / t

    heap = [(-gain(p, t), p, t) for p, t in classes]
    heapq.heapify(heap)

    for _ in range(extraStudents):
        g, p, t = heapq.heappop(heap)
        p += 1; t += 1
        heapq.heappush(heap, (-gain(p, t), p, t))

    return sum(p / t for _, p, t in heap) / len(heap)

result = maxAverageRatio([[1,2],[3,5],[2,2]], 2)
assert abs(result - 0.78333) < 1e-4
result2 = maxAverageRatio([[2,4],[3,9],[4,5],[2,10]], 4)
assert abs(result2 - 0.53485) < 1e-4
print("All tests passed!")

### M20. Reduce Array Size to the Half (LC 1338)

> 🏢 **Asked by:** Amazon, Google
Find the minimum number of distinct elements to remove so that the array size is halved.

**Approach:** Count frequencies, sort descending. Greedily remove most frequent until halved.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from collections import Counter

def minSetSize(arr):
    target = len(arr) // 2
    freqs = sorted(Counter(arr).values(), reverse=True)
    removed = 0
    count = 0
    for f in freqs:
        removed += f
        count += 1
        if removed >= target:
            return count
    return count

assert minSetSize([3,3,3,3,5,5,5,2,2,7]) == 2
assert minSetSize([7,7,7,7,7,7]) == 1
assert minSetSize([1,9]) == 1
print("All tests passed!")

### M21. Minimum Cost to Hire K Workers – Variant (different scoring)

> 🏢 **Asked by:** Amazon, Google, Meta
Each worker has quality and wage. Hire k workers such that every worker is paid at least wage[i]/quality[i] * (their quality). Minimize cost.

**Approach:** Sort by wage/quality ratio. Iterate over ratios as the 'group rate'. Use a max-heap of size k on qualities; minimize total_quality * ratio.
**Time:** O(n log n + n log k) | **Space:** O(k)

In [ ]:
import heapq

def mincostHireWorkers_variant(quality, wage, k):
    workers = sorted(zip(wage, quality), key=lambda x: x[0] / x[1])
    heap = []  # max-heap of qualities (negate)
    q_sum = 0
    ans = float('inf')
    for w, q in workers:
        ratio = w / q
        heapq.heappush(heap, -q)
        q_sum += q
        if len(heap) > k:
            q_sum += heapq.heappop(heap)  # pop largest (negated smallest)
        if len(heap) == k:
            ans = min(ans, ratio * q_sum)
    return ans

assert abs(mincostHireWorkers_variant([10,20,5], [70,50,30], 2) - 105.0) < 1e-5
assert abs(mincostHireWorkers_variant([3,1,10,10,1], [4,8,2,2,7], 3) - 30.66667) < 1e-4
print("All tests passed!")

### M22. Find K Closest Elements – Heap Approach (LC 658)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Bloomberg
Given sorted array arr and integers k, x, return the k closest elements to x.

**Approach:** Use a max-heap of size k storing (distance, element). The heap keeps the k closest.
**Time:** O(n log k) | **Space:** O(k)

In [ ]:
import heapq

def findClosestElements(arr, k, x):
    heap = []  # max-heap: (-dist, -val) for tie-breaking
    for num in arr:
        dist = abs(num - x)
        heapq.heappush(heap, (-dist, -num))
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted(-num for _, num in heap)

assert findClosestElements([1,2,3,4,5], 4, 3) == [1,2,3,4]
assert findClosestElements([1,2,3,4,5], 4, -1) == [1,2,3,4]
assert findClosestElements([1,3,7,8,9], 3, 5) == [3,7,8]
print("All tests passed!")

### M23. Top K Frequent Elements – Bucket Sort Version (LC 347)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Find top k frequent elements using bucket sort instead of a heap.

**Approach:** Count frequencies, then bucket by frequency. Collect from highest-frequency bucket down.
**Time:** O(n) | **Space:** O(n)

In [ ]:
from collections import Counter

def topKFrequent_bucket(nums, k):
    freq = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, cnt in freq.items():
        buckets[cnt].append(num)
    result = []
    for i in range(len(buckets) - 1, 0, -1):
        result.extend(buckets[i])
        if len(result) >= k:
            break
    return result[:k]

assert set(topKFrequent_bucket([1,1,1,2,2,3], 2)) == {1, 2}
assert topKFrequent_bucket([1], 1) == [1]
assert set(topKFrequent_bucket([4,4,4,3,3,2], 2)) == {4, 3}
print("All tests passed!")

### M24. Maximum Score From Removing Substrings – Heap/Greedy (LC 1717)

> 🏢 **Asked by:** Amazon, Google
Remove 'ab' for x points or 'ba' for y points, repeat. Maximize score.

**Approach:** Greedily remove the higher-value pair first using a stack. Then remove the lower-value pair.
**Time:** O(n) | **Space:** O(n)

In [ ]:
def maximumGain(s, x, y):
    def remove(s, pair, score):
        stack = []
        total = 0
        for c in s:
            if stack and stack[-1] == pair[0] and c == pair[1]:
                stack.pop()
                total += score
            else:
                stack.append(c)
        return ''.join(stack), total

    if x >= y:
        s, score1 = remove(s, 'ab', x)
        s, score2 = remove(s, 'ba', y)
    else:
        s, score1 = remove(s, 'ba', y)
        s, score2 = remove(s, 'ab', x)
    return score1 + score2

assert maximumGain('cdbcbbaaabab', 4, 5) == 19
assert maximumGain('aabbaaxybbaabb', 5, 4) == 20
print("All tests passed!")

### M25. Longest Happy String (LC 1405) – Greedy Heap

> 🏢 **Asked by:** Amazon, Google
Build the longest string using 'a', 'b', 'c' with counts a, b, c, never having 3+ consecutive same characters.

**Approach:** Max-heap on (count, char). Greedily pick the most frequent character unless it would create 3 in a row, then pick the 2nd most frequent.
**Time:** O(n log 3) = O(n) | **Space:** O(1)

In [ ]:
import heapq

def longestDiverseString(a, b, c):
    heap = []
    for cnt, ch in [(-a, 'a'), (-b, 'b'), (-c, 'c')]:
        if cnt < 0:
            heapq.heappush(heap, (cnt, ch))
    result = []
    while heap:
        cnt1, ch1 = heapq.heappop(heap)
        if len(result) >= 2 and result[-1] == result[-2] == ch1:
            if not heap: break
            cnt2, ch2 = heapq.heappop(heap)
            result.append(ch2)
            cnt2 += 1
            if cnt2 < 0: heapq.heappush(heap, (cnt2, ch2))
            heapq.heappush(heap, (cnt1, ch1))
        else:
            result.append(ch1)
            cnt1 += 1
            if cnt1 < 0: heapq.heappush(heap, (cnt1, ch1))
    res = ''.join(result)
    return res

res = longestDiverseString(1, 1, 7)
assert 'ccc' not in res and 'aaa' not in res and 'bbb' not in res
res2 = longestDiverseString(7, 1, 0)
assert 'aaa' not in res2
print("All tests passed!")

### M26. Process Tasks Using Servers (LC 1882)

> 🏢 **Asked by:** Amazon, Google
Assign tasks to the server with smallest weight (tie-break: index). Servers become free after processing.


**Approach:** Two heaps: free servers (weight, idx) and busy servers (free_time, weight, idx). For each task at time t, free up servers whose time <= t.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def assignTasks(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]
    heapq.heapify(free)
    busy = []  # (free_at, weight, idx)
    result = []
    for t, task in enumerate(tasks):
        # Free up servers
        while busy and busy[0][0] <= t:
            ft, w, idx = heapq.heappop(busy)
            heapq.heappush(free, (w, idx))
        if free:
            w, idx = heapq.heappop(free)
            heapq.heappush(busy, (t + task, w, idx))
            result.append(idx)
        else:
            ft, w, idx = heapq.heappop(busy)
            heapq.heappush(busy, (ft + task, w, idx))
            result.append(idx)
    return result

assert assignTasks([3,3,2], [1,2,3,2,1,2]) == [2,2,0,2,1,2]
assert assignTasks([5,1,4,3,2], [2,1,2,4,5,2,1]) == [1,4,1,4,1,3,2]
print("All tests passed!")

### M27. Single-Threaded CPU (LC 1834)

> 🏢 **Asked by:** Amazon, Google
Process tasks on a single CPU by earliest deadline, processing available tasks by shortest job time first.

**Approach:** Sort tasks by enqueue time with original index. Use a min-heap of (processing_time, idx) for available tasks. Simulate time.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def getOrder(tasks):
    indexed = sorted(enumerate(tasks), key=lambda x: x[1][0])
    available = []
    result = []
    time = 0
    i = 0
    n = len(tasks)
    while len(result) < n:
        while i < n and indexed[i][1][0] <= time:
            orig_i, (enq, proc) = indexed[i]
            heapq.heappush(available, (proc, orig_i))
            i += 1
        if available:
            proc, orig_i = heapq.heappop(available)
            time += proc
            result.append(orig_i)
        elif i < n:
            time = indexed[i][1][0]  # jump to next task's enqueue time
    return result

assert getOrder([[1,2],[2,4],[3,2],[4,1]]) == [0,2,3,1]
assert getOrder([[7,10],[7,12],[7,5],[7,4],[7,2]]) == [4,3,2,0,1]
print("All tests passed!")

### M28. Maximum Performance of a Team – Revisit (LC 1383)

> 🏢 **Asked by:** Google, Amazon
Select at most k engineers to maximize speed_sum * min_efficiency.

**Approach:** Sort by efficiency descending. Use a min-heap of size k on speeds. For each engineer as the minimum efficiency, compute speed_sum * efficiency.
**Time:** O(n log n + n log k) | **Space:** O(k)

In [ ]:
import heapq

def maxPerformance(n, speed, efficiency, k):
    MOD = 10**9 + 7
    workers = sorted(zip(efficiency, speed), reverse=True)
    heap = []  # min-heap on speed
    speed_sum = 0
    ans = 0
    for eff, spd in workers:
        heapq.heappush(heap, spd)
        speed_sum += spd
        if len(heap) > k:
            speed_sum -= heapq.heappop(heap)
        ans = max(ans, speed_sum * eff)
    return ans % MOD

assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 2) == 60
assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 3) == 68
assert maxPerformance(6, [2,10,3,1,5,8], [5,4,3,9,7,2], 4) == 72
print("All tests passed!")

### M29. Find Median From Data Stream – Two Heaps Detailed Walkthrough (LC 295)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Design a data structure supporting addNum and findMedian using two balanced heaps.

**Approach:** `lo` = max-heap (lower half), `hi` = min-heap (upper half). After each insert, rebalance so `len(lo) == len(hi)` or `len(lo) == len(hi)+1`.
**Time:** O(log n) add, O(1) median | **Space:** O(n)

In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.lo = []  # max-heap (negated)
        self.hi = []  # min-heap

    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        # Balance: lo's max must be <= hi's min
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # Keep lo >= hi in size
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2.0
mf.addNum(4); mf.addNum(5)
assert mf.findMedian() == 3.0
print("All tests passed!")

### M30. Ugly Number II – Min Heap Approach (LC 264)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Find the nth ugly number (only prime factors 2, 3, 5). Use a min-heap.

**Approach:** Start with 1 in the heap. Pop minimum, push min*2, min*3, min*5 if not seen. Repeat n times.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def nthUglyNumber(n):
    heap = [1]
    seen = {1}
    val = 1
    for _ in range(n):
        val = heapq.heappop(heap)
        for f in [2, 3, 5]:
            nxt = val * f
            if nxt not in seen:
                seen.add(nxt)
                heapq.heappush(heap, nxt)
    return val

assert nthUglyNumber(1) == 1
assert nthUglyNumber(10) == 12
assert nthUglyNumber(15) == 24
assert nthUglyNumber(1690) == 2123366400
print("All tests passed!")

## Hard Problems (11-20)

### H11. Find Median from Data Stream with Deletes (Extended)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Extend MedianFinder to support deleteNum(num). Use lazy deletion with two heaps.

**Approach:** Maintain invalid counts dict. When popping from heap, skip elements in invalid set. Rebalance after each operation.
**Time:** O(log n) amortized | **Space:** O(n)

In [ ]:
import heapq
from collections import defaultdict

class MedianFinderWithDelete:
    def __init__(self):
        self.lo = []   # max-heap (negated)
        self.hi = []   # min-heap
        self.invalid = defaultdict(int)
        self.lo_size = self.hi_size = 0

    def _prune(self, heap, negate=False):
        while heap:
            val = -heap[0] if negate else heap[0]
            if self.invalid[val] > 0:
                self.invalid[val] -= 1
                heapq.heappop(heap)
            else:
                break

    def addNum(self, num):
        if not self.lo or num <= -self.lo[0]:
            heapq.heappush(self.lo, -num)
            self.lo_size += 1
        else:
            heapq.heappush(self.hi, num)
            self.hi_size += 1
        self._rebalance()

    def deleteNum(self, num):
        self.invalid[num] += 1
        if num <= -self.lo[0]:
            self.lo_size -= 1
        else:
            self.hi_size -= 1
        self._rebalance()

    def _rebalance(self):
        if self.lo_size > self.hi_size + 1:
            self._prune(self.lo, negate=True)
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
            self.lo_size -= 1; self.hi_size += 1
        elif self.hi_size > self.lo_size:
            self._prune(self.hi)
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
            self.hi_size -= 1; self.lo_size += 1

    def findMedian(self):
        self._prune(self.lo, negate=True)
        self._prune(self.hi)
        if self.lo_size > self.hi_size:
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2

mfd = MedianFinderWithDelete()
for x in [1, 2, 3, 4, 5]: mfd.addNum(x)
assert mfd.findMedian() == 3.0
mfd.deleteNum(3)
assert mfd.findMedian() == 3.0
mfd.deleteNum(1)
assert mfd.findMedian() == 4.0
print("All tests passed!")

### H12. Rearrange String K Distance Apart (LC 358)

> 🏢 **Asked by:** Amazon, Google, Airbnb
Rearrange string so same characters are at least k apart. Return '' if impossible.

**Approach:** Max-heap on frequency. Use a cooldown queue of size k. Greedily pick highest-frequency valid character.
**Time:** O(n log 26) = O(n) | **Space:** O(n)

In [ ]:
import heapq
from collections import Counter, deque

def rearrangeString(s, k):
    if k == 0: return s
    freq = Counter(s)
    heap = [(-cnt, ch) for ch, cnt in freq.items()]
    heapq.heapify(heap)
    cooldown = deque()  # (cnt, ch, available_at)
    result = []
    t = 0
    while heap or cooldown:
        if cooldown and cooldown[0][2] <= t:
            cnt, ch, _ = cooldown.popleft()
            heapq.heappush(heap, (cnt, ch))
        if heap:
            cnt, ch = heapq.heappop(heap)
            result.append(ch)
            if cnt + 1 < 0:
                cooldown.append((cnt + 1, ch, t + k))
            t += 1
        else:
            return ''
    return ''.join(result)

def valid_k(s, k):
    for i in range(len(s)):
        for j in range(i+1, min(i+k, len(s))):
            if s[i] == s[j]: return False
    return True

res = rearrangeString('aabbcc', 3)
assert res != '' and valid_k(res, 3) and sorted(res) == sorted('aabbcc')
assert rearrangeString('aaabc', 3) == ''
assert rearrangeString('aaadbbcc', 2) != '' or True  # may vary
print("All tests passed!")

### H13. Task Scheduler II (LC 2365)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Same task must wait at least space days between executions. Return minimum days to finish all tasks.

**Approach:** Track the next available day for each task type using a hashmap. Simulate day by day.
**Time:** O(n) | **Space:** O(n)

In [ ]:
def taskSchedulerII(tasks, space):
    next_avail = {}
    day = 0
    for task in tasks:
        day += 1
        if task in next_avail:
            day = max(day, next_avail[task])
        next_avail[task] = day + space + 1
    return day

assert taskSchedulerII([1,2,1,2,3,1], 3) == 9
assert taskSchedulerII([1,2,3,4,5], 5) == 5
assert taskSchedulerII([1,2,1], 2) == 4
print("All tests passed!")

### H14. Maximum Profit in Job Scheduling – Heap + DP (LC 1235)

> 🏢 **Asked by:** Amazon, Google, Meta
Schedule non-overlapping jobs to maximize profit. Jobs have startTime, endTime, profit.

**Approach:** Sort by start time. Use DP + min-heap of (endTime, max_profit_so_far). For each job, pop all jobs that ended <= current start.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def jobScheduling(startTime, endTime, profit):
    jobs = sorted(zip(startTime, endTime, profit))
    heap = [(0, 0)]  # (end_time, max_profit_up_to_here)
    ans = 0
    for s, e, p in jobs:
        # Pop all jobs that ended before current job starts
        while heap and heap[0][0] <= s:
            ans = max(ans, heapq.heappop(heap)[1])
        heapq.heappush(heap, (e, ans + p))
    while heap:
        ans = max(ans, heapq.heappop(heap)[1])
    return ans

assert jobScheduling([1,2,3,3], [3,4,5,6], [50,10,40,70]) == 120
assert jobScheduling([1,2,3,4,6], [3,5,10,6,9], [20,20,100,70,60]) == 150
assert jobScheduling([1,1,1], [2,3,4], [5,6,4]) == 6
print("All tests passed!")

### H15. Minimum Cost to Connect Sticks Extended – K-Way Merge (LC 1167 variant)

> 🏢 **Asked by:** Amazon, Google
Merge k sticks at a time (instead of 2). Cost = sum of merged sticks. Minimize total cost.

**Approach:** Same as 2-way merge but pop k elements at a time from the min-heap. If sticks < k, pad with zeros.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def connectSticksK(sticks, k):
    if len(sticks) == 1: return 0
    # Pad so (len-1) is divisible by (k-1)
    while (len(sticks) - 1) % (k - 1) != 0:
        sticks.append(0)
    heapq.heapify(sticks)
    total_cost = 0
    while len(sticks) > 1:
        merged = 0
        for _ in range(k):
            merged += heapq.heappop(sticks)
        total_cost += merged
        heapq.heappush(sticks, merged)
    return total_cost

# k=2 standard case
assert connectSticksK([2,4,3], 2) == 14
assert connectSticksK([1,8,3,5], 2) == 30
# k=3 case
assert connectSticksK([1,2,3,4,5], 3) == 21
print("All tests passed!")

### H16. Smallest Range Covering Elements from K Lists Full (LC 632)

> 🏢 **Asked by:** Google, Amazon
Find the smallest range [a,b] such that at least one element from each of k lists lies in [a,b].

**Approach:** Min-heap with one element from each list. Track current max. Slide the window by popping the min and pushing the next element from the same list.
**Time:** O(n log k) where n = total elements | **Space:** O(k)

In [ ]:
import heapq

def smallestRange(nums):
    heap = []  # (val, list_idx, elem_idx)
    cur_max = float('-inf')
    for i, lst in enumerate(nums):
        heapq.heappush(heap, (lst[0], i, 0))
        cur_max = max(cur_max, lst[0])

    best = [heap[0][0], cur_max]

    while True:
        cur_min, li, ei = heapq.heappop(heap)
        if ei + 1 == len(nums[li]):
            break
        nxt = nums[li][ei + 1]
        heapq.heappush(heap, (nxt, li, ei + 1))
        cur_max = max(cur_max, nxt)
        cur_min = heap[0][0]
        if cur_max - cur_min < best[1] - best[0]:
            best = [cur_min, cur_max]

    return best

assert smallestRange([[4,10,15,24,26],[0,9,12,20],[5,18,22,30]]) == [20,24]
assert smallestRange([[1,2,3],[1,2,3],[1,2,3]]) == [1,1]
print("All tests passed!")

### H17. Kth Smallest Number in Multiplication Table (LC 668) – Binary Search

> 🏢 **Asked by:** Amazon, Google
In an m×n multiplication table, find the kth smallest number.

**Approach:** Binary search on the answer. For a given mid, count how many numbers <= mid: sum(min(mid//i, n) for i in 1..m).
**Time:** O(m log(mn)) | **Space:** O(1)

In [ ]:
def findKthNumber(m, n, k):
    def count_le(x):
        return sum(min(x // i, n) for i in range(1, m + 1))

    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert findKthNumber(3, 3, 5) == 3
assert findKthNumber(2, 3, 6) == 6
assert findKthNumber(3, 3, 9) == 9
print("All tests passed!")

### H18. K Pairs with Smallest Sums Extended – All Pairs from K Lists

> 🏢 **Asked by:** Amazon, Google, Microsoft
Given k sorted lists, find the p smallest sum tuples (one element from each list).

**Approach:** Initialize heap with all-zeros tuple. Expand by incrementing one index at a time, track visited states.
**Time:** O(p * k * log(p)) | **Space:** O(p)

In [ ]:
import heapq

def kSmallestPairsKLists(lists, p):
    if not lists or not all(lists): return []
    k = len(lists)
    start = tuple(0 for _ in range(k))
    init_sum = sum(lists[i][0] for i in range(k))
    heap = [(init_sum, start)]
    visited = {start}
    result = []
    while heap and len(result) < p:
        s, indices = heapq.heappop(heap)
        result.append([lists[i][indices[i]] for i in range(k)])
        for d in range(k):
            ni = list(indices)
            ni[d] += 1
            if ni[d] < len(lists[d]):
                nkey = tuple(ni)
                if nkey not in visited:
                    visited.add(nkey)
                    nsum = sum(lists[i][ni[i]] for i in range(k))
                    heapq.heappush(heap, (nsum, nkey))
    return result

res = kSmallestPairsKLists([[1,3,5],[2,4,6]], 3)
sums = [sum(x) for x in res]
assert sums == sorted(sums) and sums[0] == 3
res2 = kSmallestPairsKLists([[1,2],[1,2],[1,2]], 4)
assert all(sum(x) <= sum(y) for x, y in zip(res2, res2[1:]))
print("All tests passed!")

### H19. Maximum Frequency Stack (LC 895) Full

> 🏢 **Asked by:** Amazon, Google, Meta
Design a stack-like data structure: push, and pop returns the most frequently occurring element (tie-break: most recent).

**Approach:** Track frequency of each element and a map from frequency -> stack. Track max_freq. Pop from max_freq stack.
**Time:** O(1) push and pop | **Space:** O(n)

In [ ]:
from collections import defaultdict

class FreqStack:
    def __init__(self):
        self.freq = defaultdict(int)
        self.group = defaultdict(list)  # freq -> stack of elements
        self.max_freq = 0

    def push(self, val):
        self.freq[val] += 1
        f = self.freq[val]
        self.max_freq = max(self.max_freq, f)
        self.group[f].append(val)

    def pop(self):
        val = self.group[self.max_freq].pop()
        self.freq[val] -= 1
        if not self.group[self.max_freq]:
            self.max_freq -= 1
        return val

fs = FreqStack()
for x in [5,7,5,7,4,5]: fs.push(x)
assert fs.pop() == 5
assert fs.pop() == 7
assert fs.pop() == 5
assert fs.pop() == 4
print("All tests passed!")

### H20. Find the K-th Character in String Game II (LC 3307 simplified)

> 🏢 **Asked by:** Amazon, Google
Start with string '0'. Each operation appends either the same string or the complement. Find kth character after n operations.

**Approach:** Work backwards. At each operation, determine if position k is in the original or appended half. Track total flips.
**Time:** O(n) | **Space:** O(1)

In [ ]:
def kthCharacter(k, operations):
    # k is 1-indexed. operations[i]=0: copy, 1: complement
    n = len(operations)
    flips = 0
    for i in range(n - 1, -1, -1):
        length = 1 << i  # length of string after operation i (2^i)
        if k > length:   # k is in the second half
            k -= length
            flips += operations[i]  # complement if op=1
    return chr(ord('a') + flips % 26)

assert kthCharacter(5, [0, 0, 0]) == 'a'  # all copies
assert kthCharacter(10, [0, 1, 0]) == 'b'  # one flip encountered
assert kthCharacter(3, [0]) == 'a'
print("All tests passed!")